# WSJ 2027 - Adresslista för mejlutskick

Bygger ett platt MASTER-blad enligt mallen i
`input/Mall på adresslista för mejlutskick.xlsx` med kolumnerna:

| Kolumn | Källa |
|--------|-------|
| Medlemsnummer i Scoutnet | Excel MASTER (member_no) |
| Förnamn | Scoutnet API (first_name) |
| Efternamn | Scoutnet API (last_name) |
| Mejladress | Scoutnet API (primary_email) |
| Mobilnummer | Scoutnet API (contact_info.1 = deltagarens egen) |
| Avdelning | Excel MASTER (group, global 1-53) |

**Scope:** endast deltagare (1899 personer fördelade på 53 avdelningar).
Sortering: per avdelning, sedan efternamn + förnamn.

**Output:** `output/wsj27_adresslista.xlsx`

In [ ]:
import sys, importlib
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
importlib.reload(u)

import pandas as pd

INPUT_DIR = '/config/notebooks/wsj27/input'
OUTPUT_DIR = '/config/notebooks/wsj27/output'
TEMPLATE_PATH = f'{INPUT_DIR}/Mall på adresslista för mejlutskick.xlsx'
OUTPUT_PATH = f'{OUTPUT_DIR}/wsj27_adresslista.xlsx'
EXCEL_FILES = {
    'direktresa': f'{INPUT_DIR}/Direktresa Avdelningar.xlsx',
    'rundresa':   f'{INPUT_DIR}/Rundresa Avdelningar.xlsx',
}

# Läs mallens kolumnordning så outputen matchar exakt
TEMPLATE_COLUMNS = list(pd.read_excel(TEMPLATE_PATH, sheet_name='Blad1').columns)
print('Mallens kolumner:', TEMPLATE_COLUMNS)

In [ ]:
# Hämta rådata från Scoutnet (alla deltagare och övriga) - vi behöver originalfälten
raw = u.fetch_participants()
participants = raw['participants']
print(f'Hämtade {len(participants)} deltagare från Scoutnet')

In [ ]:
def extract_mobil(p):
    """Returnerar deltagarens egna mobilnummer från contact_info.1."""
    ci = p.get('contact_info') or {}
    if isinstance(ci, dict):
        return ci.get('1', '') or ''
    return ''


def build_address_rows():
    """Sammanfogar Excel-avdelning + Scoutnet-kontaktinfo till en rad per deltagare."""
    rows = []
    missing_api = []
    for path in EXCEL_FILES.values():
        xl = pd.read_excel(path, sheet_name='MASTER')
        for _, r in xl.iterrows():
            mno = int(r['member_no'])
            avd = int(r['group'])
            p = participants.get(str(mno))
            if not p:
                missing_api.append((mno, r.get('name', ''), avd))
                # Fall back till Excel-namn (split på första space)
                name = (r.get('name') or '').strip()
                if ' ' in name:
                    first, last = name.split(' ', 1)
                else:
                    first, last = name, ''
                rows.append({
                    'Medlemsnummer i Scoutnet': mno,
                    'Förnamn': first,
                    'Efternamn': last,
                    'Mejladress': '',
                    'Mobilnummer': '',
                    'Avdelning': avd,
                })
                continue
            rows.append({
                'Medlemsnummer i Scoutnet': mno,
                'Förnamn': p.get('first_name', '') or '',
                'Efternamn': p.get('last_name', '') or '',
                'Mejladress': p.get('primary_email', '') or '',
                'Mobilnummer': extract_mobil(p),
                'Avdelning': avd,
            })
    return rows, missing_api


rows, missing_api = build_address_rows()
df = pd.DataFrame(rows, columns=TEMPLATE_COLUMNS)
df = df.sort_values(['Avdelning', 'Efternamn', 'Förnamn']).reset_index(drop=True)

print(f'Totalt {len(df)} rader')
print(f'  med mejl:   {(df["Mejladress"] != "").sum()}')
print(f'  med mobil:  {(df["Mobilnummer"] != "").sum()}')
print(f'  saknas i API: {len(missing_api)}  (fall back till Excel-namn, tomma kontaktfält)')
if missing_api:
    for mno, name, avd in missing_api:
        print(f'    {mno} {name} (Avd {avd})')
print(f'\nAvdelningar: {df["Avdelning"].min()}..{df["Avdelning"].max()} '
      f'({df["Avdelning"].nunique()} unika)')
print('\nFörsta 10 raderna:')
print(df.head(10).to_string(index=False))

In [ ]:
# Skriv xlsx med samma blad-namn som mallen ('Blad1') så formatet matchar
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as w:
    df.to_excel(w, sheet_name='Blad1', index=False)
print(f'Sparade: {OUTPUT_PATH}')
print(f'  {len(df)} rader, {len(df.columns)} kolumner: {list(df.columns)}')